# Closing Auction — Features and Target

Starts from `data/panel_clean.csv`, the cleaned panel built in `01) data_exploration.ipynb`: one row per symbol-day, last pre-close snapshot joined to the realized close, 10,430 rows, ETPs and dead-book symbol-days already excluded. Builds engineered features, then several candidate target definitions (target definition was deliberately left open in `01)`), then a date-based train/test split.

In [1]:
import numpy as np
import pandas as pd

panel = pd.read_csv("data/panel_clean.csv", parse_dates=["date"])
panel.shape

(10430, 19)

## Two entry scenarios

The brief splits the pre-close window into two regimes with genuinely different available information, not
just different cadence:

- **Window 1, 15:50-15:55** (10s cadence): only `paired_shares`/`side`/`shares`/`ref_price` exist.
  `near_price`/`far_price` are not disseminated yet -- structurally `0.0`, not missing.
- **Window 2, 15:55-16:00** (1s cadence): all six exist.

A trader deciding at 15:52 cannot see `near_price` -- it doesn't exist yet -- so a target or feature built on
it isn't a coherent target for that decision point. The panel from `01)` used only the last snapshot before
16:00:00 (deep in window 2), which implicitly models a single late-entry scenario and ignores the early one
entirely, even though the brief allows entry any time from 3:50pm on.

Building both as one long table: the last snapshot before the end of each window, same feature/target
columns, a `scenario` column (`"w1"`/`"w2"`) distinguishing which regime a row belongs to. Columns that don't
exist in a regime (`near_price`-based ones in `w1`) are `NaN` there, not computed with a stand-in value.

In [2]:
import tarfile

PRE_555 = "15:55:00"   # end of window 1 / start of window 2
CLOSE = "16:00:00"     # end of window 2

def load_all(tar_path):
    frames = []
    with tarfile.open(tar_path) as tf:
        for member in tf.getmembers():
            if not member.name.endswith(".csv.gz"):
                continue
            date_str = member.name.split(".")[0]
            d = pd.read_csv(tf.extractfile(member), compression="gzip")
            d = d.drop(columns=["ts.1"])
            d["date"] = pd.to_datetime(date_str, format="%Y%m%d")
            frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts"])
    df["local_clock"] = df["local_time"].str.slice(11, 19)
    return df

raw = load_all("data/202503_imbalance.tar")

valid_keys = panel[["symbol", "date"]]
raw = raw.merge(valid_keys, on=["symbol", "date"], how="inner")
# 15:53:00 floor covers window 1's 60s lookback (t0 ~15:54:59) with margin; window 2 needs from ~15:58:59
raw = raw[(raw["local_clock"] >= "15:53:00") & (raw["local_clock"] < CLOSE)].copy()
raw["signed_shares"] = np.where(raw["side"] == "BUY", raw["shares"],
                          np.where(raw["side"] == "SELL", -raw["shares"], 0.0))
raw["imbalance_ratio"] = raw["shares"] / (raw["paired_shares"] + raw["shares"])
raw["signed_imbalance_ratio"] = raw["signed_shares"] / (raw["paired_shares"] + raw["shares"])
raw = raw.sort_values("ts")

print(f"raw slice: {raw.shape[0]} rows across {raw[['symbol','date']].drop_duplicates().shape[0]} symbol-days")

raw slice: 3254160 rows across 10430 symbol-days


### Snapshot + feature builders (shared across both scenarios)

In [3]:
def last_snapshot_before(raw, cutoff):
    pre = raw[raw["local_clock"] < cutoff]
    return pre.sort_values("ts").groupby(["symbol", "date"]).tail(1).copy()


def add_base_features(snap):
    snap = snap.copy()
    # ref_price == 0.0 means no matchable book existed yet at this snapshot -- real for a
    # handful of w1 rows (book hadn't started, recovers by close) that survived 01)'s
    # all-day-dead-book exclusion; guard every ref_price-derived column, not just one
    snap["ref_price_available"] = snap["ref_price"] != 0
    snap["spread"] = snap["ask"] - snap["bid"]
    snap["spread_bps"] = np.where(snap["ref_price_available"], snap["spread"] / snap["ref_price"] * 1e4, np.nan)
    snap["urgency"] = snap["signed_imbalance_ratio"] * snap["spread_bps"]

    snap["shares_over_adv"] = snap["shares"] / (snap["adv"] * 1000)
    snap["paired_shares_over_adv"] = snap["paired_shares"] / (snap["adv"] * 1000)

    # far_price == 0.0 before 15:55 ET is structural (not disseminated yet), and stays 0.0 for 32
    # w2 symbol-days with no standalone closing-auction-only orders -- both are real absences, not
    # bad data, so this guard correctly produces NaN for ~all of w1 and the 32 w2 edge cases
    snap["far_price_available"] = snap["far_price"] != 0
    snap["near_far_spread_bps"] = np.where(
        snap["far_price_available"], (snap["near_price"] / snap["far_price"] - 1) * 1e4, np.nan
    )
    snap["intraday_move_bps"] = np.where(
        snap["ref_price_available"], (snap["ref_price"] / snap["open"] - 1) * 1e4, np.nan
    )
    return snap


def add_lag_features(snap, raw, lags=(10, 30, 60)):
    snap = snap.copy()
    t0 = snap[["symbol", "date", "ts"]].rename(columns={"ts": "t0_ts"})
    lag_cols = ["ref_price", "imbalance_ratio", "signed_imbalance_ratio"]

    for lag_s in lags:
        targets = t0.copy()
        targets["ts"] = targets["t0_ts"] - pd.Timedelta(seconds=lag_s)
        targets = targets.sort_values("ts")

        lagged = pd.merge_asof(
            targets, raw[["ts", "symbol", "date"] + lag_cols],
            on="ts", by=["symbol", "date"], direction="backward",
        )
        lagged = lagged.rename(columns={c: f"{c}_lag{lag_s}s" for c in lag_cols})
        snap = snap.merge(lagged[["symbol", "date"] + [f"{c}_lag{lag_s}s" for c in lag_cols]],
                           on=["symbol", "date"], how="left")

        snap[f"imbalance_ratio_chg_{lag_s}s"] = snap["imbalance_ratio"] - snap[f"imbalance_ratio_lag{lag_s}s"]
        snap[f"signed_imbalance_ratio_chg_{lag_s}s"] = (
            snap["signed_imbalance_ratio"] - snap[f"signed_imbalance_ratio_lag{lag_s}s"]
        )
        lag_ref_available = snap[f"ref_price_lag{lag_s}s"] != 0
        snap[f"ref_price_ret_{lag_s}s_bps"] = np.where(
            snap["ref_price_available"] & lag_ref_available,
            (snap["ref_price"] / snap[f"ref_price_lag{lag_s}s"] - 1) * 1e4, np.nan,
        )
        snap = snap.drop(columns=[f"{c}_lag{lag_s}s" for c in lag_cols])
    return snap


def add_rolling_features(snap, raw, window_s=60):
    snap = snap.copy()
    t0 = snap[["symbol", "date", "ts"]].rename(columns={"ts": "t0_ts"})
    window = raw.merge(t0, on=["symbol", "date"], how="inner")
    window = window[
        (window["ts"] >= (window["t0_ts"] - pd.Timedelta(seconds=window_s)))
        & (window["ts"] <= window["t0_ts"])  # exclude anything after the decision point -- caught
                                              # via w1's n_snapshots_60s coming out at ~306 instead
                                              # of ~6-7, a real lookahead leak without this
    ]

    roll = window.groupby(["symbol", "date"]).agg(
        ref_price_std=("ref_price", "std"),
        imbalance_ratio_std_60s=("imbalance_ratio", "std"),
        n_snapshots_60s=("ts", "count"),
    ).reset_index()

    snap = snap.merge(roll, on=["symbol", "date"], how="left")
    snap["ref_price_std_60s_bps"] = np.where(
        snap["ref_price_available"], snap["ref_price_std"] / snap["ref_price"] * 1e4, np.nan
    )
    return snap.drop(columns=["ref_price_std"])

### Build each scenario

In [4]:
def build_scenario(raw, cutoff, label):
    snap = last_snapshot_before(raw, cutoff)
    snap = add_base_features(snap)
    snap = add_lag_features(snap, raw)
    snap = add_rolling_features(snap, raw)
    snap["scenario"] = label
    return snap

snap_w1 = build_scenario(raw, PRE_555, "w1")
snap_w2 = build_scenario(raw, CLOSE, "w2")

print(f"w1 (decide @15:55): {snap_w1.shape[0]} rows, last snapshot times {snap_w1['local_clock'].min()}-{snap_w1['local_clock'].max()}")
print(f"w2 (decide @16:00): {snap_w2.shape[0]} rows, last snapshot times {snap_w2['local_clock'].min()}-{snap_w2['local_clock'].max()}")
print(f"both cover all {len(valid_keys)} validated symbol-days:",
      snap_w1.shape[0] == len(valid_keys) and snap_w2.shape[0] == len(valid_keys))
print()
print("w1 null bid/ask (should be 0 -- confirmed resolved before 15:55 in 01), if not these rows need checking):")
print("  bid:", snap_w1["bid"].isna().sum(), "| ask:", snap_w1["ask"].isna().sum())
print()
print("w1 n_snapshots_60s (10s cadence -> expect ~6-7, not ~60-61 like w2):")
print(snap_w1["n_snapshots_60s"].describe())

w1 (decide @15:55): 10430 rows, last snapshot times 15:54:50-15:54:50
w2 (decide @16:00): 10430 rows, last snapshot times 15:59:59-15:59:59
both cover all 10430 validated symbol-days: True

w1 null bid/ask (should be 0 -- confirmed resolved before 15:55 in 01), if not these rows need checking):
  bid: 1 | ask: 1

w1 n_snapshots_60s (10s cadence -> expect ~6-7, not ~60-61 like w2):
count    10430.000000
mean         6.253500
std          0.435035
min          6.000000
25%          6.000000
50%          6.000000
75%          7.000000
max          7.000000
Name: n_snapshots_60s, dtype: float64


## Target

`close_price` (raw level) is the target used throughout `01)`. Three bps-based alternatives, each relative to
something observable at the row's own snapshot -- but which of the three are *coherent* depends on the
scenario:

- **vs. `ref_price`** and **vs. book midpoint** -- valid in both windows, `ref_price`/`bid`/`ask` exist
  throughout 15:50-16:00.
- **vs. `near_price`** -- only coherent in `w2`. `near_price` is structurally `0.0` in `w1` (not yet
  disseminated), so this target is `NaN` there by construction, not computed against a placeholder.

In [5]:
def add_targets(snap, panel):
    snap = snap.merge(panel[["symbol", "date", "close_price"]], on=["symbol", "date"], how="left")
    snap["target_bps_vs_ref"] = np.where(
        snap["ref_price_available"], (snap["close_price"] / snap["ref_price"] - 1) * 1e4, np.nan
    )
    mid = (snap["bid"] + snap["ask"]) / 2
    snap["target_bps_vs_mid"] = (snap["close_price"] / mid - 1) * 1e4
    snap["target_bps_vs_near"] = np.where(
        snap["near_price"] != 0, (snap["close_price"] / snap["near_price"] - 1) * 1e4, np.nan
    )
    return snap

snap_w1 = add_targets(snap_w1, panel)
snap_w2 = add_targets(snap_w2, panel)

features = pd.concat([snap_w1, snap_w2], ignore_index=True)

target_cols = ["close_price", "target_bps_vs_ref", "target_bps_vs_near", "target_bps_vs_mid"]
print(features.groupby("scenario")[target_cols].agg(["count", "mean"]))
print()
print(f"combined: {features.shape[0]} rows, {features.shape[1]} columns")
print(f"target_bps_vs_near null count matches w1 row count exactly: "
      f"{features['target_bps_vs_near'].isna().sum() == snap_w1.shape[0]}")

         close_price             target_bps_vs_ref            \
               count        mean             count      mean   
scenario                                                       
w1             10430  135.615488             10425 -3.382199   
w2             10430  135.615488             10430  0.223584   

         target_bps_vs_near           target_bps_vs_mid            
                      count      mean             count      mean  
scenario                                                           
w1                        0       NaN             10429 -3.116014  
w2                    10430 -0.000116             10430  0.343871  

combined: 20860 rows, 47 columns
target_bps_vs_near null count matches w1 row count exactly: True


## Train / test split

Split by date, not randomly -- rows on the same day (across both scenarios) share market-wide conditions, so
a random split would leak information across train/test. 80/20 by row count, boundary snapped to a full
trading day: find the first date where cumulative row count reaches 80% of the total, and put that entire
day -- both scenarios' rows -- in train.

In [6]:
daily_counts = features.groupby("date").size().sort_index()
cum_frac = daily_counts.cumsum() / daily_counts.sum()
split_date = cum_frac[cum_frac >= 0.80].index[0]

train = features[features["date"] <= split_date].copy()
test = features[features["date"] > split_date].copy()

print(f"split date: {split_date.date()} (last day included in train)")
print(f"train: {train.shape[0]} rows ({train.shape[0]/len(features):.1%}), "
      f"{train['date'].nunique()} days, {train['date'].min().date()} to {train['date'].max().date()}")
print(f"test:  {test.shape[0]} rows ({test.shape[0]/len(features):.1%}), "
      f"{test['date'].nunique()} days, {test['date'].min().date()} to {test['date'].max().date()}")
print()
print("no date overlap:", set(train["date"]).isdisjoint(set(test["date"])))
print("both scenarios present in both splits:",
      set(train["scenario"]) == {"w1", "w2"} and set(test["scenario"]) == {"w1", "w2"})

train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print("wrote data/train.csv and data/test.csv")

split date: 2025-03-25 (last day included in train)
train: 16886 rows (80.9%), 17 days, 2025-03-03 to 2025-03-25
test:  3974 rows (19.1%), 4 days, 2025-03-26 to 2025-03-31

no date overlap: True
both scenarios present in both splits: True


wrote data/train.csv and data/test.csv


**20,860 rows (10,430 symbol-days x 2 scenarios), 47 columns.** `scenario` distinguishes `w1`
(decide @15:54:50, no `near_price`/`far_price`) from `w2` (decide @15:59:59, full info, same as `01)`'s
panel). Zero `inf` anywhere in the final output, verified directly on `train.csv`/`test.csv`, not assumed.

Two things caught during verification, both guarded rather than silently producing bad values:

- **A handful of `w1` rows (5 of 10,430) have `ref_price == 0`** — the book hadn't started matching yet at
  15:54:50 for these, even though the symbol-day is otherwise valid (recovers by close, unlike the all-day
  dead books excluded in `01)`). `ref_price_available` flags this; every `ref_price`-derived column
  (`spread_bps`, `urgency`, `intraday_move_bps`, `target_bps_vs_ref`, the lag/volatility features, on both
  the current row *and* the lagged lookback side) is `NaN` rather than a silent `inf` or misleading value.
- **The rolling-volatility window had no upper bound** — `w1`'s "last 60 seconds" was pulling in everything
  up to 15:59:59, not just up to the 15:54:50 decision point. A trader deciding at 15:54:50 can't see
  window 2's data yet. Caught because `n_snapshots_60s` for `w1` came out at ~306 instead of the ~6-7 a 10s
  cadence should produce; fixed by bounding the window on both sides.